# Batch 5B — Rev. 4 BF16 Generative Baseline

This notebook runs the registered `rq2_bf16_baseline_v1` study. It reruns only the 36-row Rev. 4 free-form generative baseline in true BF16. The frozen model revision, prompt, decoding settings, questions, ordered evidence windows, ODP policy, and verifier remain fixed; the frozen 4-bit baseline and ComplianceGPT outputs are unchanged references.

Before selecting **Runtime → Run all**:

1. Select **Runtime → Change runtime type → A100 GPU**. An L4 is also valid if A100 is unavailable. A T4 is not valid for this study because it lacks native BF16 support.
2. Approve the Google Drive mount. It is deliberately the first code cell so every checkpoint is persistent from the beginning.
3. Keep the output directory name `MyDrive/rq2_bf16_baseline_v1`. Do not reuse it for another model, precision, code version, or experiment.

The runner checkpoints after every question and safely resumes after a disconnect. It verifies the exact frozen RQ2 v3 inputs and source code, the pinned Hugging Face revision, all model parameter dtypes, and the absence of quantization before inference.

In [8]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_DIR = Path('/content/drive/MyDrive/rq2_bf16_baseline_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent checkpoint directory:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent checkpoint directory: /content/drive/MyDrive/rq2_bf16_baseline_v1


In [9]:
import torch

assert torch.cuda.is_available(), (
    'CUDA GPU not found. Choose Runtime > Change runtime type > A100 GPU, then rerun.'
)
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
assert torch.cuda.is_bf16_supported(), (
    f'{gpu_name} has no native BF16 path. T4 is not valid for Batch 5B; choose A100 '
    '(preferred) or L4.'
)
print(f'BF16-capable GPU: {gpu_name} ({gpu_memory_gb:.1f} GiB)')
if 'A100' not in gpu_name.upper():
    print('A100 is preferred. This notebook will continue because native BF16 is available.')

BF16-capable GPU: NVIDIA A100-SXM4-40GB (39.5 GiB)


In [10]:
import shutil
import subprocess

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = 'c83d0cd4297513979d746639a1684a1fea03a7f1'
FROZEN_SOURCE_COMMIT = 'be862bcadfa61b474d795303e01ce9394909fdcc'
REPO_DIR = Path('/content/ComplianceGPT')

# /content is disposable; the Drive checkpoint directory is never removed here.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.mkdir(parents=True)
subprocess.run(['git', 'init', '-q'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=REPO_DIR, check=True)
subprocess.run(
    ['git', 'fetch', '-q', '--depth', '1', 'origin', SOURCE_COMMIT],
    cwd=REPO_DIR, check=True,
)
subprocess.run(['git', 'checkout', '-q', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
# The runner reconstructs the exact frozen RQ2 v3 source tree from this commit.
subprocess.run(
    ['git', 'fetch', '-q', '--depth', '1', 'origin', FROZEN_SOURCE_COMMIT],
    cwd=REPO_DIR, check=True,
)
checked_out_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert checked_out_commit == SOURCE_COMMIT, (checked_out_commit, SOURCE_COMMIT)
subprocess.run(
    ['git', 'cat-file', '-e', f'{FROZEN_SOURCE_COMMIT}^{{commit}}'],
    cwd=REPO_DIR, check=True,
)
runner = REPO_DIR / 'experiments/answerer_comparison/rq2_bf16_baseline/run_bf16_baseline.py'
assert runner.is_file(), f'Batch 5B runner is missing: {runner}'
print('Batch 5B source commit:', checked_out_commit)
print('Frozen RQ2 source commit available:', FROZEN_SOURCE_COMMIT)
print('Runner:', runner)

Batch 5B source commit: c83d0cd4297513979d746639a1684a1fea03a7f1
Frozen RQ2 source commit available: be862bcadfa61b474d795303e01ce9394909fdcc
Runner: /content/ComplianceGPT/experiments/answerer_comparison/rq2_bf16_baseline/run_bf16_baseline.py


In [11]:
import importlib.metadata
import sys

requirements = REPO_DIR / 'experiments/answerer_comparison/rq2_matched/requirements-colab.txt'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', '-r', str(requirements)],
    check=True,
)
for package in ('transformers', 'accelerate', 'huggingface-hub', 'numpy'):
    print(f'{package}: {importlib.metadata.version(package)}')

transformers: 4.57.3
accelerate: 1.12.0
huggingface-hub: 0.36.0
numpy: 2.0.2


In [12]:
import hashlib
import json
from IPython.display import Markdown, display

FROZEN_ARCHIVE_SHA256 = '56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328'
frozen_archive = REPO_DIR / 'experiments/answerer_comparison/rq2_matched/results_v3/compliancegpt_rq2_matched_v3.zip'
actual_archive_sha256 = hashlib.sha256(frozen_archive.read_bytes()).hexdigest()
assert actual_archive_sha256 == FROZEN_ARCHIVE_SHA256, (
    actual_archive_sha256, FROZEN_ARCHIVE_SHA256
)
print('Frozen RQ2 v3 archive verified:', actual_archive_sha256)

prepare_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--prepare-only',
]
subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
preflight = json.loads((OUTPUT_DIR / 'preflight/summary.json').read_text(encoding='utf-8'))
assert preflight.get('scope') == 'gold-free frozen-context integrity audit; no model inference'
assert preflight.get('context_rows') == 36, preflight
assert preflight.get('distinct_context_hashes') == 36, preflight
assert preflight.get('distinct_window_hashes') == 36, preflight
print('Prepared-context audit passed: 36 immutable Rev. 4 contexts.')
display(Markdown((OUTPUT_DIR / 'PREPARED_CONTEXTS.md').read_text(encoding='utf-8')))

Frozen RQ2 v3 archive verified: 56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328
Prepared-context audit passed: 36 immutable Rev. 4 contexts.


# Batch 5B Prepared-Context Audit

This is a gold-free integrity check, not a completed BF16 result.

- Result identity: `rq2_bf16_baseline_v1`
- Frozen Rev. 4 contexts: 36
- Distinct context hashes: 36
- Distinct evidence-window hashes: 36
- Context SHA-256: `4f1ca834993a34c22bb4b61fa9a984d32382b5e84e3473656a22af688412d7a6`
- Model inference performed: no


In [13]:
from huggingface_hub import HfApi

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
MODEL_REVISION = 'a09a35458c702b33eeacc393d103063234e8bc28'
model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION)
assert model_info.sha == MODEL_REVISION, (model_info.sha, MODEL_REVISION)
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), (
    'The BF16-capable CUDA runtime was lost before model loading.'
)
print(f'Pinned model verified: {MODEL_ID}@{model_info.sha}')
print('GPU ready:', torch.cuda.get_device_name(0))

Pinned model verified: Qwen/Qwen2.5-7B-Instruct@a09a35458c702b33eeacc393d103063234e8bc28
GPU ready: NVIDIA A100-SXM4-40GB


In [14]:
import os
from collections import deque

run_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
run_environment = os.environ.copy()
run_environment.setdefault('TOKENIZERS_PARALLELISM', 'false')
run_environment.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
tail = deque(maxlen=400)
print('Starting the 36-row Rev. 4 BF16 generative-baseline run.')
print('After a disconnect, reconnect and Run all; validated completed rows are skipped.')
process = subprocess.Popen(
    run_command, cwd=REPO_DIR, env=run_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    tail.append(line)
return_code = process.wait()
failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
if return_code != 0:
    failure_log.write_text(''.join(tail), encoding='utf-8')
    print(f'Batch 5B failed. The final runner output is saved at {failure_log}.')
    raise subprocess.CalledProcessError(return_code, run_command)
if failure_log.exists():
    failure_log.unlink()
print('Batch 5B runner completed successfully.')

Starting the 36-row Rev. 4 BF16 generative-baseline run.
After a disconnect, reconnect and Run all; validated completed rows are skipped.
Frozen Batch 5B inputs verified: 36 Rev. 4 contexts.
[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib
2026-09-29 13:18:42.778640: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-29 13:18:42.846929: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Loading checkpoint shards: 100%|██████████| 4/4 [

In [15]:
import csv

contract_path = OUTPUT_DIR / 'contracts/rev4_generative_baseline_bf16.csv'
with contract_path.open('r', encoding='utf-8', newline='') as handle:
    completed_rows = list(csv.DictReader(handle))
assert len(completed_rows) == 36, f'Expected 36 completed rows, found {len(completed_rows)}'
assert {row['system'] for row in completed_rows} == {'generative_baseline_bf16'}

run_config = json.loads((OUTPUT_DIR / 'run_config.json').read_text(encoding='utf-8'))
runtime = json.loads((OUTPUT_DIR / 'manifests/bf16_runtime.json').read_text(encoding='utf-8'))
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
assert run_config.get('result_id') == 'rq2_bf16_baseline_v1'
assert run_config.get('precision') == 'bfloat16'
assert run_config.get('load_in_4bit') is False
dtype_counts = runtime.get('floating_parameter_tensor_dtypes', {})
assert set(dtype_counts) == {'torch.bfloat16'} and dtype_counts['torch.bfloat16'] > 0, dtype_counts
assert not any(runtime.get('quantization_flags', {}).values()), runtime.get('quantization_flags')
assert summary.get('n_questions') == 36
assert summary.get('matched_input_validation', {}).get('evidence_window_mismatches') == 0

result_zip = OUTPUT_DIR.with_suffix('.zip')
assert result_zip.is_file(), f'Result archive is missing: {result_zip}'
result_zip_sha256 = hashlib.sha256(result_zip.read_bytes()).hexdigest()
print('Validated BF16 rows:', len(completed_rows))
print('Result archive:', result_zip)
print('Result archive SHA-256:', result_zip_sha256)
display(Markdown((OUTPUT_DIR / 'SUMMARY.md').read_text(encoding='utf-8')))

Validated BF16 rows: 36
Result archive: /content/drive/MyDrive/rq2_bf16_baseline_v1.zip
Result archive SHA-256: 8717b3ce3c691bc47d61ee62ec88fe3038ca170636e0f868e7720ff7a754eed2


# Batch 5B — Rev. 4 BF16 Generative Baseline

Result identity: `rq2_bf16_baseline_v1`

Only the free-form baseline's model-weight precision changes from 4-bit to BF16. The model revision, prompt, decoding, questions, ordered evidence windows, ODP policy, and verifier are held fixed.

| System | Strict pass | Full clause coverage | Runtime contract pass | Mean clause precision | Mean selected clauses | ODP sensitivity* |
|---|---:|---:|---:|---:|---:|---:|
| Generative baseline, BF16 | 8/36 (0.222) | 20/36 (0.556) | 14/36 (0.389) | 0.575 | 2.89 | 0/19 (0.000) |
| Generative baseline, 4-bit (frozen) | 8/36 (0.222) | 18/36 (0.500) | 13/36 (0.361) | 0.488 | 3.33 | 0/19 (0.000) |
| ComplianceGPT, 4-bit (frozen) | 29/36 (0.806) | 29/36 (0.806) | 36/36 (1.000) | 0.505 | 4.03 | 19/19 (1.000) |

## Paired strict-pass comparisons

| Comparison (left vs right) | Left only | Right only | Exact McNemar p |
|---|---:|---:|---:|
| BF16 baseline vs 4-bit baseline | 2 | 2 | 1 |
| BF16 baseline vs 4-bit ComplianceGPT | 0 | 21 | 9.5367432e-07 |

## Interpretation boundary

This study isolates model-weight precision within the same Qwen2.5-7B free-form baseline. It is neither a frontier-model comparison nor a retraining study. A BF16 change estimates quantization sensitivity on these 36 fixed Rev. 4 rows.

*ODP operating characteristics use author labels and remain provisional until blinded independent annotation is returned.*


In [16]:
# Optional: change to True and run this cell to download a local copy.
DOWNLOAD_RESULT_ZIP = False
if DOWNLOAD_RESULT_ZIP:
    from google.colab import files
    files.download(str(result_zip))
else:
    print('Result remains safely stored in Google Drive:', result_zip)

Result remains safely stored in Google Drive: /content/drive/MyDrive/rq2_bf16_baseline_v1.zip
